# Imports and Setup

In [1]:
import os
import io
import sys
import time
import pandas as pd
from tqdm import tqdm
import anthropic
from concurrent.futures import ProcessPoolExecutor
from dotenv import load_dotenv
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM

/home/eqp6pg/.conda/envs/my-torch/lib/python3.8/site-packages/deepeval/__init__.py:51: UserWarning: You are using deepeval version 2.0, however version 2.7.0 is available. You should consider upgrading via the "pip install --upgrade deepeval" command.
  warnings.warn(


In [2]:
os.cpu_count()

128

# Claude connection

In [3]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [4]:
# Define a custom Claude model for deepeval
class Claude(DeepEvalBaseLLM):
    """Class to implement Claude model for DeepEval"""
    def __init__(self, model, api_key):
        self.model = model
        self.api_key = api_key
        self.client = anthropic.Anthropic(api_key=self.api_key)
    
    def load_model(self):
        return self.model
    
    def generate(self, prompt, max_tokens=4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    
    async def a_generate(self, prompt, max_tokens=4096):
        response = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    
    def get_model_name(self):
        return "Claude Model"

In [5]:
claude_model = "claude-3-haiku-20240307"
claude_instance = Claude(model=claude_model, api_key=anthropic_key)

# Dataset

In [6]:
df = pd.read_csv("data/data.csv")

# Create Deepeval test cases
test_cases = [
    LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"]
    )
    for _, row in df.iterrows()
]

In [7]:
test_cases = test_cases[1000:1600]

In [8]:
len(test_cases)

600

# Metrics

In [9]:
# Define the four G-Eval metrics
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the collective quality of all sentences in the summary. The summary should be well-structured and well-organized and should build a coherent body of information about a topic.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=claude_instance,
    async_mode=False
)

consistency_metric = GEval(
    name="Consistency",
    criteria="Evaluate the factual alignment between the summary and source document. The summary should contain only statements that are entailed by the source document.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=claude_instance,
    async_mode=False
)

fluency_metric = GEval(
    name="Fluency",
    criteria="Evaluate the quality of individual sentences of the summary. The summary should have no formatting problems and grammatical errors that make the summary difficult to read.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=claude_instance,
    async_mode=False
)

relevance_metric = GEval(
    name="Relevance",
    criteria="Evaluate the selection of the most important content from the source document. The summary should include only important information from the source document.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=claude_instance,
    async_mode=False
)

# Evaluation

In [10]:
# Function to evaluate a single case with all metrics
def evaluate_case_safe(test_case, index):
    try:
        # Suppress output for each worker process
        original_stdout = sys.stdout
        sys.stdout = io.StringIO()
        
        # Evaluate with each metric
        coherence_metric.measure(test_case)
        consistency_metric.measure(test_case)
        fluency_metric.measure(test_case)
        relevance_metric.measure(test_case)
        
        # Get scores and reasons
        coherence_score = coherence_metric.score
        consistency_score = consistency_metric.score
        fluency_score = fluency_metric.score
        relevance_score = relevance_metric.score
        
        # Calculate an average score
        average_score = (coherence_score + consistency_score + fluency_score + relevance_score) / 4
        
        # Restore stdout
        sys.stdout = original_stdout
        
        return {
            "index": index,
            "input": test_case.input[:10] + "...",
            "coherence_score": coherence_score,
            "consistency_score": consistency_score,
            "fluency_score": fluency_score,
            "relevance_score": relevance_score,
            "average_score": average_score,
            "success": True
        }
    except Exception as e:
        # Restore stdout on error
        sys.stdout = original_stdout
        return {
            "index": index,
            "input": test_case.input[:10] + "...",
            "coherence_score": None,
            "consistency_score": None,
            "fluency_score": None,
            "relevance_score": None,
            "average_score": None,
            "success": False,
            "error": str(e)
        }

In [11]:
# Start timer
start_time = time.time()

# Get available CPU cores and set max workers
cpu_cores = os.cpu_count()
max_workers = min(cpu_cores, 5)

# Suppress main process output
original_stdout = sys.stdout
sys.stdout = io.StringIO()

# Parallel evaluation with tqdm and error handling
results = []
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(evaluate_case_safe, test_cases[i], i): i for i in range(len(test_cases))}
    
    # Restore stdout for progress bar
    sys.stdout = original_stdout
    
    for future in tqdm(futures, total=len(test_cases), desc="Evaluating Summaries with G-Eval"):
        result = future.result()
        results.append(result)

# End timer
end_time = time.time()
elapsed_time = end_time - start_time

Evaluating Summaries with G-Eval: 100%|██████████| 600/600 [10:46<00:00,  1.08s/it]  


In [12]:
# Sort results by index to maintain original order
results.sort(key=lambda x: x["index"])

# Create a DataFrame from results and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv("geval_summarization_scores_1000-1600.csv", index=False)

In [13]:
# Print summary
print(f"\nEvaluation completed in {elapsed_time:.2f} seconds")
print(f"Processed {len(test_cases)} test cases")
print(f"Successful evaluations: {sum(1 for r in results if r['success'])}")
print(f"Failed evaluations: {sum(1 for r in results if not r['success'])}")
print(f"Results saved to: geval_summarization_scores.csv")


Evaluation completed in 649.61 seconds
Processed 600 test cases
Successful evaluations: 587
Failed evaluations: 13
Results saved to: geval_summarization_scores.csv


In [14]:
# Print a sample of scores if available
successful_results = [r for r in results if r["success"]]
if successful_results:
    avg_coherence = sum(r["coherence_score"] for r in successful_results) / len(successful_results)
    avg_consistency = sum(r["consistency_score"] for r in successful_results) / len(successful_results)
    avg_fluency = sum(r["fluency_score"] for r in successful_results) / len(successful_results)
    avg_relevance = sum(r["relevance_score"] for r in successful_results) / len(successful_results)
    overall_avg = sum(r["average_score"] for r in successful_results) / len(successful_results)
    
    print(f"\nAverage scores across all successful evaluations:")
    print(f"  Coherence:   {avg_coherence:.4f}")
    print(f"  Consistency: {avg_consistency:.4f}")
    print(f"  Fluency:     {avg_fluency:.4f}")
    print(f"  Relevance:   {avg_relevance:.4f}")
    print(f"  Overall:     {overall_avg:.4f}")


Average scores across all successful evaluations:
  Coherence:   0.7165
  Consistency: 0.7705
  Fluency:     0.7644
  Relevance:   0.7009
  Overall:     0.7381
